# Expressing the 0.4.0 spike model in the PR&nbsp;#164 (`main`) functional form

This notebook shows that the manuscript's **multidms 0.4.0** global-epistasis-with-shifts model
and the current **`main` (PR&nbsp;#164)** model are one forward model in two coordinate systems, and
that the transformation between them **preserves the Huber loss**.

We do *not* use the fitted pickle. Instead we read the sigmoid global-epistasis shape parameters
approximately from the committed diagnostic plot `ge_fits.png`, and take the per-mutation effects /
shifts and the observed functional scores from the committed spike-analysis CSVs
(`matsengrp/SARS-CoV-2_spike_multidms` @ `6c98b7b`).

**Three checks:**
1. the 0.4.0 and `main` Huber losses coincide (exactly as `D_d → 0`);
2. recomputed predicted-vs-observed correlations match `func_score_corr.png` (Pearson r ≈ 0.80–0.90);
3. `main`'s own `functional_score_loss` reproduces the main-form loss.

## The transformation

Let $\phi_d(v)$ be the latent phenotype of variant $v$ in condition $d$ and $\sigma$ the logistic sigmoid.

**0.4.0** (identity output, $\gamma=0$): latent $\phi_d(v)=\beta_0+\alpha_d+X_v\,(\beta+\Delta_d)$,
GE $g(z)=\theta_\text{scale}\,\sigma(z)+\theta_\text{bias}$, and the Huber loss is fit to the **raw**
prediction $\hat y_d(v)=\theta_\text{scale}\,\sigma(\phi_d(v))+\theta_\text{bias}$ (WT subtracted only when reporting).

**`main`**: latent $\phi_d(v)=\beta_{0,d}+X_v\,\beta_d$, parameter-free GE $g(z)=\sigma(z)$, and the loss
is fit to the **wildtype-subtracted** prediction $\hat y_d(v)=\alpha\,(\sigma(\phi_d(v))-\sigma(\phi_{\mathrm{wt},d}))$.

**Map (0.4.0 → main):**

| `main` parameter | from 0.4.0 |
|---|---|
| $\beta_d = \beta+\Delta_d$ | per-condition absolute effect vector (reference: $\Delta=0\Rightarrow\beta_\text{ref}=\beta$) |
| $\beta_{0,d} = \beta_0+\alpha_d$ | fold the additive offset into the per-condition intercept |
| $\alpha = \theta_\text{scale}$ | the sigmoid *range* becomes `main`'s output scale |

With these, $\phi_d(v)$ is identical variant-for-variant, so the two Huber residuals differ only by a
per-condition constant

$$\text{residual}_{0.4.0}-\text{residual}_\text{main}=\theta_\text{bias}+\theta_\text{scale}\,\sigma(\phi_{\mathrm{wt},d})\equiv D_d .$$

`main` has no free per-condition output constant (only the structural $-\alpha\,\sigma(\phi_{\mathrm{wt},d})$),
so the losses coincide exactly when $D_d=0$ — i.e. when the fit puts each condition's WT at raw score $0$.
In `ge_fits.png` the WT vertical lines sit on the $g(\phi)=0$ crossing, i.e. $D_d\approx0$.

In [1]:
import os
import numpy as np
import pandas as pd
import scipy.sparse
import multidms
import multidms.jaxmodels as jm
import jax.numpy as jnp

HERE = os.getcwd()   # run this notebook from experiments/v04-equivalence/
DATA = os.path.join(HERE, "data")

REFERENCE   = "Omicron_BA1"
CHOSEN_LASSO = 4e-5
REPLICATE    = "rep-2"     # committed label in mutations_df.csv
REPLICATE_NUM = 2         # matching value in training_functional_scores.csv
DELTA = 1.0               # Huber delta (0.4.0 and main default)

# ---- sigmoid GE-shape parameters read approximately from ge_fits.png ----
THETA_SCALE    = 7.4      # sigmoid range (midpoint read; upper asymptote off-screen)
THETA_BIAS_RAW = -3.4     # lower asymptote
# WT vertical lines cluster near phi = -0.4 for all conditions (replicate-2 panel)
PHI_WT_READ = {"Delta": -0.4, "Omicron_BA1": -0.4, "Omicron_BA2": -0.4}

sigmoid = lambda z: 1.0 / (1.0 + np.exp(-z))
def huber(r, delta=DELTA):
    a = np.abs(r)
    return np.where(a <= delta, 0.5 * a * a, delta * (a - 0.5 * delta))

## Load committed data and build the reference-frame encoding

In [2]:
# Fetch the committed spike-analysis CSVs (pinned commit) if not already present locally.
import urllib.request
os.makedirs(DATA, exist_ok=True)
BASE = ("https://raw.githubusercontent.com/matsengrp/SARS-CoV-2_spike_multidms/"
        "6c98b7b607d7387b508cdaa192d659ee9fca7367/results/spike_analysis")
for _f in ["mutations_df.csv", "training_functional_scores.csv"]:
    _p = os.path.join(DATA, _f)
    if not os.path.exists(_p):
        print("downloading", _f)
        urllib.request.urlretrieve(f"{BASE}/{_f}", _p)

# observed functional scores (one replicate), collapsing identical variants by mean
# (matches the fit's collapse_identical_variants="mean")
fs = pd.read_csv(os.path.join(DATA, "training_functional_scores.csv"))
fs = fs[fs["replicate"] == REPLICATE_NUM].copy()
fs["aa_substitutions"] = fs["aa_substitutions"].fillna("")
variants = fs.groupby(["condition", "aa_substitutions"], as_index=False)["func_score"].mean()
print(f"variants: raw={len(fs)} collapsed={len(variants)}")

mdata = multidms.Data(variants, reference=REFERENCE,
                      alphabet=multidms.AAS_WITHSTOP_WITHGAP, assert_site_integrity=False)
muts = list(mdata.mutations)
idx = {m: i for i, m in enumerate(muts)}
conditions = list(mdata.conditions)
print(f"n_mutations={len(muts)}  conditions={conditions}  reference={mdata.reference}")

# per-mutation beta / shifts (chosen lasso + replicate)
md_ = pd.read_csv(os.path.join(DATA, "mutations_df.csv"))
md_ = md_[(md_["dataset_name"] == REPLICATE)
          & np.isclose(md_["scale_coeff_lasso_shift"].astype(float), CHOSEN_LASSO)].set_index("mutation")
beta = np.zeros(len(muts)); shift = {c: np.zeros(len(muts)) for c in conditions}
for m in muts:
    if m in md_.index:
        beta[idx[m]] = md_.at[m, "beta"]
        shift["Delta"][idx[m]] = md_.at[m, "shift_Delta"]
        shift["Omicron_BA2"][idx[m]] = md_.at[m, "shift_Omicron_BA2"]
beta_d = {c: beta + shift[c] for c in conditions}

def cond_arrays(c):
    X = mdata.arrays["X"][c]                       # BCOO, WT at row 0
    Xs = scipy.sparse.csr_array((np.asarray(X.data),
        (np.asarray(X.indices[:, 0]), np.asarray(X.indices[:, 1]))), shape=X.shape)
    x_wt = np.asarray(Xs[[0], :].todense()).ravel()
    return Xs[1:], x_wt, np.asarray(mdata.arrays["y"][c])[1:]

# intercept from the read WT latent:  phi_wt = beta0 + x_wt . beta_d
beta0 = {}
for c in conditions:
    _, x_wt, _ = cond_arrays(c)
    beta0[c] = PHI_WT_READ[c] - float(x_wt @ beta_d[c])

THETA_BIAS_CAL = -THETA_SCALE * sigmoid(PHI_WT_READ[REFERENCE])   # enforce D_ref = 0
print(f"theta_bias: raw={THETA_BIAS_RAW}  calibrated(D_ref=0)={THETA_BIAS_CAL:.4f}")

variants: raw=333910 collapsed=149006


n_mutations=10870  conditions=['Delta', 'Omicron_BA1', 'Omicron_BA2']  reference=Omicron_BA1


theta_bias: raw=-3.4  calibrated(D_ref=0)=-2.9697


## Check 1 — the Huber loss is preserved by the transformation

In [3]:
def loss_table(theta_bias):
    rows = []
    for c in conditions:
        Xv, x_wt, y = cond_arrays(c)
        phi = beta0[c] + np.asarray(Xv @ beta_d[c]).ravel()
        phi_wt = beta0[c] + float(x_wt @ beta_d[c])
        D_d = theta_bias + THETA_SCALE * sigmoid(phi_wt)
        pred_040  = THETA_SCALE * sigmoid(phi) + theta_bias
        pred_main = THETA_SCALE * (sigmoid(phi) - sigmoid(phi_wt))
        rows.append(dict(condition=c, D_d=D_d,
                         loss_0_4_0=float(huber(pred_040 - y).mean()),
                         loss_main=float(huber(pred_main - y).mean())))
    t = pd.DataFrame(rows)
    t["abs_gap"] = (t["loss_0_4_0"] - t["loss_main"]).abs()
    return t

print("Raw eyeballed theta_bias = %.3f" % THETA_BIAS_RAW)
display(loss_table(THETA_BIAS_RAW).round(6))
print("\nCalibrated theta_bias = %.4f  (D_ref = 0)" % THETA_BIAS_CAL)
display(loss_table(THETA_BIAS_CAL).round(6))

Raw eyeballed theta_bias = -3.400


,condition,D_d,loss_0_4_0,loss_main,abs_gap
0,Delta,-0.430289,0.324947,0.289200,0.035747
1,Omicron_BA1,-0.430289,0.243844,0.290895,0.047050
2,Omicron_BA2,-0.430289,0.238204,0.236814,0.001390



Calibrated theta_bias = -2.9697  (D_ref = 0)


,condition,D_d,loss_0_4_0,loss_main,abs_gap
0,Delta,0.0,0.289200,0.289200,0.0
1,Omicron_BA1,0.0,0.290895,0.290895,0.0
2,Omicron_BA2,0.0,0.236814,0.236814,0.0


When `theta_bias` is calibrated so $D_d=0$, the 0.4.0 and `main` Huber losses are **identical to
machine precision**. With the raw eyeballed value the per-condition constant $D_d$ is small and the
loss gap is correspondingly tiny — exactly the behaviour the transformation predicts.

## Check 2 — predicted-vs-observed correlation matches `func_score_corr.png`

In [4]:
# Pearson r within a condition is invariant to affine transforms of the prediction, so it is
# identical for the 0.4.0 and main forms and validates the latent-phenotype reconstruction.
print("target (func_score_corr.png): Delta ~0.80, Omicron_BA1 ~0.86-0.90, Omicron_BA2 ~0.86-0.88\n")
for c in conditions:
    Xv, x_wt, y = cond_arrays(c)
    phi = beta0[c] + np.asarray(Xv @ beta_d[c]).ravel()
    phi_wt = beta0[c] + float(x_wt @ beta_d[c])
    pred_040  = THETA_SCALE * sigmoid(phi) + THETA_BIAS_CAL
    pred_main = THETA_SCALE * (sigmoid(phi) - sigmoid(phi_wt))
    m = np.isfinite(pred_040) & np.isfinite(y)
    r040  = np.corrcoef(pred_040[m], y[m])[0, 1]
    rmain = np.corrcoef(pred_main[m], y[m])[0, 1]
    print(f"  {c:16s} r(0.4.0)={r040:.3f}  r(main)={rmain:.3f}  (n={int(m.sum())})")

target (func_score_corr.png): Delta ~0.80, Omicron_BA1 ~0.86-0.90, Omicron_BA2 ~0.86-0.88

  Delta            r(0.4.0)=0.795  r(main)=0.795  (n=27190)
  Omicron_BA1      r(0.4.0)=0.891  r(main)=0.891  (n=57531)
  Omicron_BA2      r(0.4.0)=0.876  r(main)=0.876  (n=52280)


## Check 3 — `main`'s own `functional_score_loss`

In [5]:
data_sets = {c: jm.Data.from_multidms(mdata, c) for c in conditions}
model = jm.Model(
    φ={c: jm.Latent(β0=jnp.array(float(beta0[c])), β=jnp.asarray(beta_d[c])) for c in conditions},
    α=jnp.array(float(THETA_SCALE)),
    logθ={c: jnp.array(0.0) for c in conditions},
    reference_condition=REFERENCE,
    global_epistasis=jm.Sigmoid(),
)
loss = jm.functional_score_loss(model, data_sets, δ=DELTA)
for c in conditions:
    Xv, x_wt, y = cond_arrays(c)
    phi = beta0[c] + np.asarray(Xv @ beta_d[c]).ravel()
    phi_wt = beta0[c] + float(x_wt @ beta_d[c])
    direct = float(huber(THETA_SCALE * (sigmoid(phi) - sigmoid(phi_wt)) - y).mean())
    fscl = float(loss[c]); n = len(y)
    print(f"  {c:16s} functional_score_loss={fscl:.4f}  direct_mean={direct:.6f}  fscl/n={fscl/n:.6f}")
# NOTE: the current `main` uses .mean(); some builds sum per condition, hence fscl/n == direct_mean.

  Delta            functional_score_loss=0.2892  direct_mean=0.289200  fscl/n=0.000011
  Omicron_BA1      functional_score_loss=0.2909  direct_mean=0.290895  fscl/n=0.000005
  Omicron_BA2      functional_score_loss=0.2368  direct_mean=0.236814  fscl/n=0.000005


## Worked example: one real reference-homolog variant, end-to-end

A variant of the reference homolog **Omicron_BA1**, `T632N A846V` (replicate 2), observed
functional score **y = −0.375**. Reference variants have no homolog "bundle" (the wildtype is the
all-zero vector) and zero shift ($\Delta_\text{BA1}=0\Rightarrow\beta_d=\beta$), so the arithmetic
is short. Calibrated shape: $\theta_\text{scale}=7.4$, $\theta_\text{bias}=-2.97$, and
$\beta_{0,\text{BA1}}=\phi_{\mathrm{wt}}=-0.40$.

**Step 1 — per-condition effect vector $\beta_d = \beta + \Delta_d$** (reference: $\Delta=0$):

| mutation | $\beta$ | $\Delta_\text{BA1}$ | $\beta_\text{BA1}$ |
|---|---|---|---|
| T632N | −0.166 | 0 | −0.166 |
| A846V | −0.267 | 0 | −0.267 |

**Step 2 — latent phenotype.** Each pipeline assembles the latent phenotype $\phi$ (the position on
the global-epistasis curve) from the mutations the variant carries, bookkept differently but
arriving at the *same* number:

- **0.4.0** works from one effect vector $\beta$ *shared across all homologs* plus this homolog's
  shift $\Delta_d$: $\phi = \beta_0 + \alpha_d + \sum(\beta + \Delta_d)$. For the reference the
  shift is zero and $\alpha_d$ folds away, so $\phi = -0.40 + (-0.166 - 0.267)$.
- **main** works from this homolog's *own* absolute effect vector $\beta_d$ and intercept
  $\beta_{0,d}$: $\phi = \beta_{0,d} + \sum \beta_d$. For the reference $\beta_\text{BA1}=\beta$ and
  $\beta_{0,\text{BA1}}=-0.40$, giving the same $\phi = -0.40 + (-0.166 - 0.267)$.

Either way $\phi = -0.833$, so $\sigma(\phi) = 0.303$; the wildtype sits at $\phi_\mathrm{wt}=-0.40$,
$\sigma(\phi_\mathrm{wt}) = 0.401$.

**Step 3 — pass the latent phenotype through the global-epistasis function.** Both pipelines feed
$\phi$ through the sigmoid global-epistasis function $\sigma$ to get a predicted functional score,
differing only in how that output is scaled and referenced:

- **0.4.0** applies its two-parameter sigmoid directly: $\hat y = \theta_\text{scale}\,\sigma(\phi)
  + \theta_\text{bias}$ (raw curve output; wildtype subtracted only when reporting).
- **main** applies the parameter-free sigmoid, subtracts the wildtype's sigmoid output *inside* the
  prediction, and scales by $\alpha$: $\hat y = \alpha\,(\sigma(\phi) - \sigma(\phi_\mathrm{wt}))$.

| framework | formula | value |
|---|---|---|
| 0.4.0 (raw sigmoid) | $7.4\cdot0.303 - 2.97$ | **−0.727** |
| main (WT-subtracted) | $7.4\,(0.303 - 0.401)$ | **−0.727** |

Identical, because $\theta_\text{bias}$ is cancelled by main's $-\alpha\,\sigma(\phi_\mathrm{wt})$
term ($D_\text{BA1} = 0$).

**Step 4 — Huber loss for this variant** ($\delta = 1$): residual $= \hat y - y = -0.727 - (-0.375)
= -0.352$, so Huber $= \tfrac12(0.352)^2 = 0.062$ in **both** forms.

*(For a non-reference homolog every variant's reference-frame encoding also carries that homolog's
bundle of defining mutations; the bundle enters both $\phi$ and $\phi_\mathrm{wt}$ and cancels in
the WT-subtraction, so predictions reflect a variant's effect relative to its own homolog's WT.
The code uses that full encoding — e.g. Delta variant rows carry ~32 reference-frame mutations,
BA.2 ~16, BA.1 ~2.)*

## Generating the `main`-model initialization parameters

The same parameter map can be *materialized* as CSV files that fully specify a `main`
`jaxmodels.Model`, so the 0.4.0 fit can be re-initialized in the new form without a pkl. The
helper `convert_040_to_main_params.py` applies the map and writes three CSVs under
`main_init_params/`:

- `main_params_beta.csv` &mdash; long `(condition, mutation, beta)` &rarr; `Model.φ[c].β`
- `main_params_latent.csv` &mdash; per condition `beta0`, `logtheta`, `is_reference` (plus
  `phi_wt`, `D_d` diagnostics) &rarr; `Model.φ[c].β0`, `Model.logθ[c]`
- `main_params_global.csv` &mdash; `alpha`, `reference_condition`, `global_epistasis=Sigmoid`,
  `output_activation=IdentityOutput`, `huber_delta`, calibrated `theta_bias` &rarr; `Model.α`,
  `Model.reference_condition`, and the structural choices

The cell below drives that converter from the objects already built in this notebook, then
rebuilds a `Model` *purely from the CSVs* and confirms its per-condition Huber loss matches the
in-memory model to machine precision &mdash; i.e. the CSVs initialize `main` at the intended
point in parameter space. (`logθ_d = 0` and the calibrated `theta_bias` are recorded for
completeness; neither enters `functional_score_loss`, whose WT-subtraction reproduces the
0.4.0 loss.)

In [6]:
# Drive the standalone converter (same directory) from this notebook's objects.
import sys
sys.path.insert(0, HERE)
import convert_040_to_main_params as conv

x_wt_all = {c: cond_arrays(c)[1] for c in conditions}          # reference-frame WT bundle per condition
params = conv.convert_040_to_main(
    conditions, beta_d, x_wt_all, PHI_WT_READ,
    theta_scale=THETA_SCALE, theta_bias_raw=THETA_BIAS_RAW, reference=REFERENCE,
)
paths = conv.write_csvs(params, muts, conditions)
print("wrote:")
for p in paths:
    print("  ", os.path.relpath(p, HERE))

# Rebuild a jaxmodels.Model purely from the CSVs and compare to the in-memory model (cell above).
model_csv, _ = conv.build_model_from_csvs(mdata)
loss_csv = jm.functional_score_loss(model_csv, data_sets, δ=DELTA)
print(f"\n{'condition':16s} {'β0(csv)':>10s} {'loss(csv)':>12s} {'loss(mem)':>12s} {'|gap|':>10s}")
for c in conditions:
    lc, lm = float(loss_csv[c]), float(loss[c])
    print(f"{c:16s} {float(model_csv.φ[c].β0):+10.4f} {lc:12.6f} {lm:12.6f} {abs(lc - lm):10.2e}")

wrote:
   main_init_params/main_params_beta.csv
   main_init_params/main_params_latent.csv
   main_init_params/main_params_global.csv



condition           β0(csv)    loss(csv)    loss(mem)      |gap|
Delta               -0.4524     0.289200     0.289200   5.55e-17
Omicron_BA1         -0.4000     0.290895     0.290895   0.00e+00
Omicron_BA2         -0.5614     0.236814     0.236814   0.00e+00
